# vLLM Quickstart 한국어 튜토리얼

공식 vLLM 저장소(`vllm-project/vllm`) 태그 **v0.30.0** 기준 공식 예제를 번역+해설한 튜토리얼입니다.
원본: `examples/basic/offline_inference/basic.py`, `examples/basic/online_serving/openai_chat_completion_client.py`

**권장 런타임**: GPU(L4 권장, T4는 최근 알려진 버그로 불안정함). 메뉴 → 런타임 유형 변경 → 하드웨어 가속기(L4).

## 이 튜토리얼에서 배우는 것
1. offline 추론(`LLM` 클래스로 바로 프롬프트 넣고 결과 받기)
2. OpenAI 호환 서버 기동하고 API로 호출하기
3. attention 백엔드 자동 선택 결과를 DEBUG 로그로 역추론하는 법

## 유의점 (미리 읽어두면 사고가 안 생김)
- **`pip install vllm`이 torch를 새 버전으로 끌어올리면서 Colab 기본 torchaudio와 CUDA 버전이 어긋나 `RuntimeError`가 날 수 있습니다.** 설치 셀에서 torchaudio를 함께 제거해 회피합니다(텍스트 전용 튜토리얼이라 불필요).
- **offline 추론은 별도 python 프로세스 스크립트로 실행합니다.** Jupyter 커널 안에서 `LLM(...)`을 직접 부르면 (a) 워커가 Jupyter의 가짜 stdout을 물려받아 `io.UnsupportedOperation: fileno`로 죽거나, (b) 죽지 않아도 `llm` 객체가 GPU 메모리를 계속 물고 있어 다음 셀(서버 기동)이 메모리를 못 얻고 실패합니다 — 둘 다 직접 겪은 문제입니다. 별도 프로세스로 돌리면 끝나는 순간 GPU 메모리가 자동 해제돼 **커널 재시작이 필요 없습니다.**
- 서버는 백그라운드로 띄우고 `/health`를 폴링해야 합니다. 포그라운드로 띄우면 셀 실행이 끝나지 않습니다.
- 실습이 끝나면 **런타임을 바로 종료**하세요 — L4는 켜져 있는 동안 계속 Colab 컴퓨트 유닛을 소진합니다.


## 0. 환경 확인
재현성을 위해 먼저 GPU/드라이버 정보를 남깁니다.

In [ ]:
!nvidia-smi
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"

## 1. vLLM 설치 (v0.30.0 고정)
이 튜토리얼은 버전 변화에 흔들리지 않도록 태그를 고정합니다.

In [ ]:
!pip install -q vllm==0.30.0 openai
# transformers가 import 시 torchaudio를 함께 로드하는데, vllm이 끌어올린 torch(CUDA 13.0)와
# Colab 기본 torchaudio(CUDA 12.8)의 버전이 어긋나 RuntimeError가 납니다.
# 이 튜토리얼은 텍스트 전용이라 torchaudio 자체가 불필요 — 제거해서 충돌을 피합니다.
!pip uninstall -q -y torchaudio

## 2. Offline 추론
원본: `examples/basic/offline_inference/basic.py` (v0.30.0)

`LLM` 클래스가 모델을 로드하고, `generate()`가 프롬프트 리스트를 받아 한 번에 배치 실행합니다.

In [ ]:
%%writefile /content/offline_inference.py
# offline 추론을 별도 프로세스 스크립트로 분리한 이유(중요):
# vLLM의 LLM(...)을 Jupyter 커널 안에서 직접 호출하면 두 가지 문제가 있습니다.
#  1) 워커 프로세스가 Jupyter의 가짜 stdout(fileno 없음)을 물려받아
#     `io.UnsupportedOperation: fileno`로 죽을 수 있습니다.
#  2) LLM 객체가 셀 실행이 끝나도 커널에 남아 GPU 메모리를 계속 점유합니다 —
#     실제로 nvidia-smi로 확인해 보면 `VLLM::EngineCore` 프로세스가 20GB 넘게 물고
#     있는 채로 다음 셀들이 실행돼, 그다음 LLM() 호출이 메모리를 못 얻고
#     "Engine core initialization failed"로 조용히 실패합니다.
# 별도 python 프로세스로 실행하면 두 문제 다 사라집니다: 프로세스가 끝나면 GPU 메모리가
# 자동으로 해제되고, stdout도 평범한 파이프라 fileno 문제가 없습니다.
import os

os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"

from vllm import LLM, SamplingParams

prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]
sampling_params = SamplingParams(temperature=0.8, top_p=0.95)

llm = LLM(model="facebook/opt-125m")
outputs = llm.generate(prompts, sampling_params)

print("\n생성 결과:\n" + "-" * 60)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt:    {prompt!r}")
    print(f"Output:    {generated_text!r}")
    print("-" * 60)

In [ ]:
!python /content/offline_inference.py > /content/offline.log 2>&1; \
echo "=== 생성 결과 ==="; \
sed -n '/생성 결과/,$p' /content/offline.log; \
echo ""; \
echo "=== attention backend 선택 근거 (DEBUG 로그) ==="; \
grep -E "Some attention backends are not valid for" -A5 /content/offline.log || echo "(해당 줄 없음 — 아래 tail 참고)"; \
echo ""; \
echo "=== 에러 발생 시 마지막 40줄 ==="; \
grep -qE "Traceback|Error" /content/offline.log && tail -40 /content/offline.log || echo "(에러 없음)"

### 어떤 attention 백엔드가 선택되었나? (정정)
공식 문서/여러 블로그는 "시작 로그에 `Using X attention backend`가 찍힌다"고 설명하지만,
v0.30.0 소스(`platforms/cuda.py`)를 직접 보면 그 `logger.info("Using %s backend.", ...)` 줄은
**`--attention-backend`로 백엔드를 직접 지정했을 때만** 실행됩니다. 우리처럼 아무것도 지정하지
않고 자동 선택에 맡기면 결과는 `logger.debug_once(...)`로만 남습니다 — 그래서 스크립트를
`VLLM_LOGGING_LEVEL=DEBUG`로 띄우고, 위 셀에서 그 로그를 `grep`으로 뽑았습니다.

`Some attention backends are not valid for ...`로 시작하는 줄의 `Reasons: {...}` 안에는
**탈락한 backend와 이유만** 나열됩니다 — 선택된 backend는 그 목록에 없는 것으로 역으로
추론해야 합니다(vLLM이 "이게 선택됨"이라고 INFO로 따로 찍어주지 않기 때문). 선택 우선순위
자체는 `vllm/platforms/cuda.py`의 `_get_backend_priorities()`에 있고, 대체로 FLASH_ATTN/
FLASHINFER가 우선 시도되고, 그 둘 중 현재 설정을 지원하지 못하면 TRITON_ATTN으로 폴백합니다.
L4(SM89)는 bf16/fp8까지 지원하는 비교적 새 아키텍처라 대부분 FLASH_ATTN이 선택됩니다 —
단 위 셀의 실제 `Reasons` 목록으로 반드시 확인하세요.

## 4. OpenAI 호환 서버 기동
원본: `examples/basic/online_serving/openai_chat_completion_client.py` (v0.30.0)

서버는 백그라운드 프로세스로 띄웁니다. 포그라운드로 띄우면 셀 실행이 영원히 끝나지 않습니다.

In [ ]:
import subprocess
import time
import urllib.request

log_path = "/content/vllm_server.log"
log_file = open(log_path, "w")

server_proc = subprocess.Popen(
    ["vllm", "serve", "facebook/opt-125m"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print(f"서버 PID: {server_proc.pid}, 로그: {log_path}")

# /health를 폴링해서 서버가 뜨기까지 기다립니다(모델 로드 포함 최대 몇 분 걸릴 수 있음).
for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("서버 준비 완료")
        break
    except Exception:
        time.sleep(5)
else:
    print("타임아웃 — 로그를 확인하세요:")
    print(open(log_path).read()[-2000:])

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="EMPTY", base_url="http://localhost:8000/v1")

models = client.models.list()
model = models.data[0].id

# 주의: facebook/opt-125m은 채팅용으로 파인튜닝되지 않은 순수 base 모델이라
# 채팅 템플릿이 없습니다. 그래서 원본 예제의 chat.completions.create()를 그대로
# 쓰면 "you must provide a chat template" 400 에러가 납니다(transformers v4.44부터
# 기본 채팅 템플릿 자동 적용이 없어졌기 때문). base 모델에는 completions API(=
# 순수 텍스트 이어쓰기)가 맞는 사용법입니다.
completion = client.completions.create(
    model=model,
    prompt="Who won the world series in 2020?",
    max_tokens=50,
)
print(completion)

## 5. 종료 (반드시 실행)
자원(GPU/컴퓨트 유닛)을 아끼기 위해 서버 프로세스를 종료하고, 이 셀을 마지막으로 실행한 후 런타임도 종료하세요.

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=30)
print("서버 종료됨")